<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/websocket_chat_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WebSocket チャットデモ (Google Colab)

`websockets` ライブラリで **チャットサーバ** を立て、**ngrok** で公開して、
PC やスマホの複数ブラウザから同時にチャットできるデモです。

**構成**

```
[ブラウザ A] --wss--\
[ブラウザ B] --wss----> ngrok --> Colab上の websockets サーバ (1ポート)
[ブラウザ C] --wss--/                 ├ GET /      -> チャット画面(HTML)を返す
                                      └ /ws        -> WebSocket でメッセージ中継
```

- サーバはメッセージを受け取ると **接続中の全員にブロードキャスト**します。
- 同じ 1 ポートで HTML 配信と WebSocket を兼用するので、**ngrok トンネルは 1 本**で済みます（無料プラン向き）。
- 上から順にセルを実行してください。

In [1]:
!pip install -q websockets==12.0 pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 12.0 which is incompatible.
langsmith 0.8.12 requires websockets>=15.0, but you have websockets 12.0 which is incompatible.
gradio-client 1.14.0 requires websockets<16.0,>=13.0, but you have websockets 12.0 which is incompatible.
google-genai 1.68.0 requires websockets<17.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
dataproc-spark-connect 1.1.0 requires websockets>=14.0, but you have websockets 12.0 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 12.0 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 12.0 which is incompatible.


## 1. チャット画面(HTML/JS クライアント) を定義
バニラ JS・ダークテーマ・日本語UI。`window.__WS_URL__` があればそこへ、無ければ同一ホストの `/ws` へ自動接続します。

In [2]:
HTML_PAGE = r'''<!doctype html>
<html lang="ja">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>WebSocket Chat Demo</title>
<style>
  :root{
    --bg:#0d1117; --panel:#161b22; --panel2:#1c2330;
    --border:#30363d; --text:#e6edf3; --muted:#8b949e;
    --accent:#39d3bb; --accent2:#58a6ff; --me:#1f6feb; --sys:#6e7681;
  }
  *{box-sizing:border-box}
  html,body{margin:0;height:100%}
  body{
    background:var(--bg); color:var(--text);
    font-family:"Hiragino Kaku Gothic ProN","Yu Gothic",Meiryo,system-ui,sans-serif;
    display:flex; flex-direction:column; height:100vh;
  }
  header{
    padding:10px 16px; border-bottom:1px solid var(--border);
    background:var(--panel); display:flex; align-items:center; gap:12px;
  }
  header h1{font-size:15px; margin:0; letter-spacing:.5px;
    font-family:"SFMono-Regular",Consolas,Menlo,monospace; color:var(--accent);}
  .status{font-size:12px; color:var(--muted); display:flex; align-items:center; gap:6px; margin-left:auto;}
  .dot{width:9px;height:9px;border-radius:50%;background:var(--sys);transition:background .3s}
  .dot.on{background:var(--accent)} .dot.off{background:#f85149}
  .count{font-size:12px;color:var(--accent2);font-family:monospace}

  #log{flex:1; overflow-y:auto; padding:14px 16px; display:flex; flex-direction:column; gap:8px;}
  .msg{max-width:78%; padding:8px 12px; border-radius:12px; line-height:1.5; font-size:14px;
       background:var(--panel2); border:1px solid var(--border); align-self:flex-start; word-break:break-word;}
  .msg.me{align-self:flex-end; background:var(--me); border-color:var(--me);}
  .msg .name{font-size:11px; color:var(--accent); margin-bottom:2px; font-weight:600;}
  .msg.me .name{color:#cfe1ff;}
  .msg .time{font-size:10px; color:var(--muted); margin-left:8px;}
  .msg.me .time{color:#cfe1ff;}
  .system{align-self:center; font-size:12px; color:var(--sys); background:transparent; border:none; padding:2px;}

  footer{padding:10px 12px; border-top:1px solid var(--border); background:var(--panel);
         display:flex; gap:8px; align-items:flex-end;}
  .namebox{display:flex; flex-direction:column; gap:3px;}
  .namebox label{font-size:10px;color:var(--muted)}
  input,button{font-family:inherit; font-size:14px;}
  #name{width:120px; padding:8px; border-radius:8px; border:1px solid var(--border);
        background:var(--bg); color:var(--text);}
  #text{flex:1; padding:10px 12px; border-radius:8px; border:1px solid var(--border);
        background:var(--bg); color:var(--text);}
  #text:focus,#name:focus{outline:none;border-color:var(--accent2)}
  #send{padding:10px 18px; border-radius:8px; border:none; cursor:pointer;
        background:var(--accent); color:#06231f; font-weight:700;}
  #send:disabled{opacity:.4; cursor:not-allowed}

  /* ログイン画面 */
  #login{position:fixed; inset:0; z-index:50; background:rgba(13,17,23,.94);
         display:flex; align-items:center; justify-content:center;}
  .login-card{background:var(--panel); border:1px solid var(--border); border-radius:14px;
              padding:28px 26px; width:300px; text-align:center;
              display:flex; flex-direction:column; gap:14px;}
  .login-card h2{margin:0; color:var(--accent); font-family:monospace; font-size:18px;}
  .login-card p{margin:0; color:var(--muted); font-size:13px;}
  .login-card input{padding:10px 12px; border-radius:8px; border:1px solid var(--border);
                    background:var(--bg); color:var(--text); font-size:15px; text-align:center;}
  .login-card input:focus{outline:none; border-color:var(--accent2);}
  .login-card button{padding:11px; border-radius:8px; border:none; cursor:pointer;
                     background:var(--accent); color:#06231f; font-weight:700; font-size:15px;}
  .login-card button:disabled{opacity:.4; cursor:not-allowed;}
  .hidden{display:none !important;}
</style>
</head>
<body>
  <div id="login">
    <div class="login-card">
      <h2>&#128172; WebSocket Chat</h2>
      <p>名前を入力して参加してください</p>
      <input id="loginName" placeholder="名前" maxlength="20" autocomplete="off">
      <button id="loginBtn" disabled>参加する</button>
    </div>
  </div>
  <header>
    <h1>&#128172; WebSocket Chat</h1>
    <div class="status">
      <span class="dot" id="dot"></span><span id="state">未参加</span>
      <span class="count" id="count"></span>
    </div>
  </header>
  <div id="log"></div>
  <footer>
    <div class="namebox">
      <label for="name">名前</label>
      <input id="name" placeholder="名無し" maxlength="20">
    </div>
    <input id="text" placeholder="メッセージを入力 (Enter で送信)" autocomplete="off">
    <button id="send" disabled>送信</button>
  </footer>

<script>
  // 接続先: インライン表示時は window.__WS_URL__ を注入。
  // ngrok の URL を直接ブラウザで開いた場合は同一ホストへ自動接続。
  const WS_URL = window.__WS_URL__ ||
    ((location.protocol === "https:" ? "wss://" : "ws://") + location.host + "/ws");

  const log = document.getElementById("log");
  const dot = document.getElementById("dot");
  const stateEl = document.getElementById("state");
  const countEl = document.getElementById("count");
  const nameEl = document.getElementById("name");
  const textEl = document.getElementById("text");
  const sendBtn = document.getElementById("send");

  let ws;
  const CLIENT_ID = (window.crypto && crypto.randomUUID)
        ? crypto.randomUUID() : ("id-" + Math.random().toString(36).slice(2));
  const curName = () => (nameEl.value || "名無し").trim() || "名無し";

  function esc(s){return s.replace(/[&<>"']/g, c =>
    ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));}

  function addMsg(name, text, time, mine){
    const d = document.createElement("div");
    d.className = "msg" + (mine ? " me" : "");
    d.innerHTML = '<div class="name">'+esc(name)+'<span class="time">'+esc(time)+'</span></div>'+esc(text);
    log.appendChild(d); log.scrollTop = log.scrollHeight;
  }
  function addSys(text){
    const d = document.createElement("div");
    d.className = "system"; d.textContent = text;
    log.appendChild(d); log.scrollTop = log.scrollHeight;
  }

  function connect(){
    stateEl.textContent = "接続中...";
    ws = new WebSocket(WS_URL);
    ws.onopen = () => {
      dot.className = "dot on"; stateEl.textContent = "接続済み";
      sendBtn.disabled = false;
      ws.send(JSON.stringify({type:"join", name: curName(), id: CLIENT_ID}));
    };
    ws.onmessage = (e) => {
      const m = JSON.parse(e.data);
      if(m.type === "message") addMsg(m.name, m.text, m.time, m.id === CLIENT_ID);
      else if(m.type === "system") addSys(m.text + "  (" + m.time + ")");
      else if(m.type === "count") countEl.textContent = "オンライン: " + m.count;
    };
    ws.onclose = () => {
      dot.className = "dot off"; stateEl.textContent = "切断 — 3秒後に再接続";
      sendBtn.disabled = true; setTimeout(connect, 3000);
    };
    ws.onerror = () => ws.close();
  }

  function send(){
    const t = textEl.value.trim();
    if(!t || !ws || ws.readyState !== 1) return;
    ws.send(JSON.stringify({type:"chat", name: curName(), text: t, id: CLIENT_ID}));
    textEl.value = ""; textEl.focus();
  }

  // 名前欄を変えたら rename を送る（フォーカスを外す / Enter で確定）
  nameEl.addEventListener("change", () => {
    if(ws && ws.readyState === 1)
      ws.send(JSON.stringify({type:"rename", name: curName(), id: CLIENT_ID}));
  });
  nameEl.addEventListener("keydown", e => { if(e.key === "Enter") nameEl.blur(); });

  sendBtn.onclick = send;
  textEl.addEventListener("keydown", e => { if(e.key === "Enter") send(); });

  // --- ログイン: 名前を入れて「参加する」で初めて接続する ---
  const loginEl = document.getElementById("login");
  const loginNameEl = document.getElementById("loginName");
  const loginBtn = document.getElementById("loginBtn");
  loginNameEl.addEventListener("input", () => {
    loginBtn.disabled = loginNameEl.value.trim() === "";
  });
  function doLogin(){
    const n = loginNameEl.value.trim();
    if(!n) return;
    nameEl.value = n;                 // フッターの名前欄へ反映（以後はここで改名可）
    loginEl.classList.add("hidden");
    connect();
    textEl.focus();
  }
  loginBtn.onclick = doLogin;
  loginNameEl.addEventListener("keydown", e => { if(e.key === "Enter") doLogin(); });
  loginNameEl.focus();
</script>
</body>
</html>
'''
print('HTML クライアントを読み込みました (%d 文字)' % len(HTML_PAGE))


HTML クライアントを読み込みました (8194 文字)


## 2. WebSocket サーバを起動
バックグラウンドスレッドで動かすのでセルはすぐ返ります。接続/切断/発言はこのセルの下にログ表示されます。

In [3]:
import asyncio, json, threading, http
from datetime import datetime
import websockets

PORT = 8765
USERS = {}        # websocket -> 表示名
_started = False   # 二重起動ガード

def now():
    return datetime.now().strftime("%H:%M:%S")

def jdump(obj):
    return json.dumps(obj, ensure_ascii=False)

def push_count():
    websockets.broadcast(list(USERS.keys()), jdump({"type": "count", "count": len(USERS)}))

# --- WebSocket ハンドラ: 1接続 = 1クライアント ---
async def handler(ws):
    USERS[ws] = None
    print(f"[{now()}] 接続  (現在 {len(USERS)} 人)")
    try:
        async for raw in ws:
            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                continue
            t = data.get("type")
            if t == "join":
                USERS[ws] = (data.get("name") or "名無し").strip() or "名無し"
                websockets.broadcast(list(USERS.keys()),
                    jdump({"type": "system", "text": f"{USERS[ws]} さんが参加しました", "time": now()}))
                push_count()
            elif t == "rename":
                old = USERS.get(ws) or "名無し"
                new = (data.get("name") or "名無し").strip() or "名無し"
                if new != old:
                    USERS[ws] = new
                    websockets.broadcast(list(USERS.keys()),
                        jdump({"type": "system",
                               "text": f"{old} さんが {new} に名前を変更しました", "time": now()}))
            elif t == "chat":
                # chat に乗ってきた名前が現在の名前と違えば、改名として扱う
                old = USERS.get(ws) or "名無し"
                new = (data.get("name") or "名無し").strip() or "名無し"
                if new != old:
                    USERS[ws] = new
                    websockets.broadcast(list(USERS.keys()),
                        jdump({"type": "system",
                               "text": f"{old} さんが {new} に名前を変更しました", "time": now()}))
                msg = {"type": "message", "name": USERS[ws],
                       "text": str(data.get("text", "")), "time": now(),
                       "id": data.get("id")}
                websockets.broadcast(list(USERS.keys()), jdump(msg))
                print(f"[{now()}] {USERS[ws]}: {msg['text']}")
    finally:
        name = USERS.pop(ws, None)
        print(f"[{now()}] 切断  (現在 {len(USERS)} 人)")
        if name:
            websockets.broadcast(list(USERS.keys()),
                jdump({"type": "system", "text": f"{name} さんが退室しました", "time": now()}))
        push_count()

# --- 同じポートで HTML(GET /) と WebSocket(/ws) を兼用 ---
# websockets==12.0 の旧 process_request シグネチャ: (path, request_headers)
async def process_request(path, request_headers):
    if path == "/ws":
        return None                      # None を返すと WebSocket ハンドシェイクに進む
    body = HTML_PAGE.encode("utf-8")     # それ以外は HTML を返す（普通のブラウザGET）
    headers = [("Content-Type", "text/html; charset=utf-8"),
               ("Content-Length", str(len(body)))]
    return http.HTTPStatus.OK, headers, body

async def _main():
    async with websockets.serve(handler, "0.0.0.0", PORT, process_request=process_request):
        print(f"WebSocket サーバ起動: ws://0.0.0.0:{PORT}/ws")
        await asyncio.Future()           # 永久に走らせる

def _run():
    asyncio.run(_main())

if not _started:
    threading.Thread(target=_run, daemon=True).start()
    _started = True
    print("サーバをバックグラウンドスレッドで起動しました。")
else:
    print("既に起動済みです。コードを変えた場合はランタイムを再起動してください。")


サーバをバックグラウンドスレッドで起動しました。


## 3. ngrok で公開
初回は authtoken の設定が必要です。出力された **公開URL** を学生に配ってください。

In [5]:
# ngrok の authtoken を設定（https://dashboard.ngrok.com/get-started/your-authtoken で取得）
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "2HhWtFPzcpgWIRTD67dJ7y59sYx_3nvbFfQgmH2JPJ7Aevg4t" # ここに自分の authtoken を貼る
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 既存トンネルを掃除してから 1本だけ張る（無料プランは同時1本）
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(PORT, "http")
public_https = tunnel.public_url                       # 例: https://xxxx.ngrok-free.app
public_wss   = public_https.replace("https://", "wss://")
print("公開URL (学生はこれをブラウザで開く):", public_https)
print("WebSocket URL :", public_wss + "/ws")


公開URL (学生はこれをブラウザで開く): https://bc87-34-11-36-223.ngrok-free.app
WebSocket URL : wss://bc87-34-11-36-223.ngrok-free.app/ws


## 4. 自分用にノートブック内で表示
まずここで動作確認。学生は手順3の公開URLを各自のブラウザ/スマホで開きます。

In [6]:
from IPython.display import HTML, display

# インライン表示用に WS の接続先を明示注入（Colab出力はiframe隔離のため location.host が使えない）
inline = ('<script>window.__WS_URL__="' + public_wss + '/ws";</script>') + HTML_PAGE
display(HTML(inline))


## 使い方・補足

- **学生の参加**: 手順3で出た `https://....ngrok-free.app` を各自のブラウザで開き、**名前を入力して「参加する」**を押すと入室します。以降の発言は全員に届きます。
- **ngrok 無料プランの警告ページ**: 初回アクセス時に「Visit Site」の確認画面が出ます。一度クリックすれば通過します。
- **複数人テスト**: 自分のスマホとPCで同じURLを開けば 1人でも複数接続を確認できます。
- **作り直すとき**: サーバのコードを変更したら、メニューの「ランタイム → ランタイムを再起動」してから上から実行し直してください（ポート二重起動を避けるため）。
- **プロトコル観察**: ブラウザの開発者ツール → Network → WS を開くと、`join` / `chat` の JSON フレームが流れる様子を確認できます。授業で見せると分かりやすいです。

### メッセージ形式 (JSON)
| 向き | type | 内容 |
|---|---|---|
| client → server | `join` | `{type, name, id}` 入室 |
| client → server | `rename` | `{type, name, id}` 改名 |
| client → server | `chat` | `{type, name, text, id}` 発言 |
| server → clients | `message` | `{type, name, text, time, id}` 発言の中継 |
| server → clients | `system` | `{type, text, time}` 入退室通知 |
| server → clients | `count` | `{type, count}` 接続人数 |